# Nhiệm vụ 1.1: Phân tích Phân phối Sản phẩm, Thị phần Thương hiệu & Nguồn bán

**Thành viên thực hiện**: TV3 (Exploratory Data Analysis)

**Tập dữ liệu đầu vào**: `data/processed/products_processed.jsonl` (10.316 listing, 30 thuộc tính chuẩn hóa)

**Mục tiêu**:
1. Khám phá cơ cấu dữ liệu theo 5 nguồn cào (`source`) và 2 ngành hàng (`category`: Laptop & Smartphone).
2. Phân tích thị phần các thương hiệu hàng đầu (`brand_norm`) toàn thị trường và so sánh đối đầu giữa Laptop và Smartphone.
3. Đánh giá mức độ tập trung của các nhà bán lẻ (`seller_raw`).

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Import các module EDA chuẩn hóa
sys.path.append(str(Path.cwd().parent / "scripts"))
sys.path.append("src/eda/scripts")

from eda_stats import (
    load_processed_products,
    get_source_category_crosstab,
    get_brand_market_share,
    get_seller_distribution
)
from eda_viz import (
    set_plotting_theme,
    plot_source_category_distribution,
    plot_brand_market_share_comparison,
    plot_seller_share
)

set_plotting_theme()
print("Môi trường phân tích sẵn sàng!")

## 1. Tải dữ liệu và kiểm tra quy mô quan sát

In [ ]:
DATA_PATH = "data/processed/products_processed.jsonl"
df = load_processed_products(DATA_PATH)
print(f"Tổng số quan sát listing: {len(df):,}")
print(f"Số lượng thuộc tính: {df.shape[1]}")
df[["listing_record_id", "product_name", "source", "category", "brand_norm", "price_vnd"]].head()

## 2. Phân bố sản phẩm theo Nguồn bán (Source) và Ngành hàng (Category)

In [ ]:
ct_counts, ct_pct = get_source_category_crosstab(df)
print("--- Số lượng quan sát theo Source x Category ---")
display(ct_counts)

print("\n--- Tỷ trọng % so với toàn bộ thị trường ---")
display(ct_pct)

In [ ]:
# Vẽ biểu đồ phân bố nguồn và danh mục
fig_path = "src/eda/outputs/figures/fig_1_1_source_category_distribution.png"
plot_source_category_distribution(df, fig_path)

# Hiển thị inline
plt.figure(figsize=(9, 5))
sns.barplot(
    data=df,
    x="source",
    hue="category",
    order=df["source"].value_counts().index,
    palette={"laptop": "#2b5c8f", "smartphone": "#d95f02"},
    estimator=len
)
plt.title("Phân bố số lượng sản phẩm theo Nguồn cào & Ngành hàng")
plt.show()

## 3. Phân tích Thị phần Thương hiệu (Brand Market Share)

In [ ]:
brand_dict = get_brand_market_share(df, top_n=10)

print("Top 10 Hãng Laptop:")
display(brand_dict["laptop"])

print("\nTop 10 Hãng Smartphone:")
display(brand_dict["smartphone"])

In [ ]:
# Biểu đồ so sánh Top thương hiệu Laptop vs Smartphone
fig_brand_path = "src/eda/outputs/figures/fig_1_1_brand_market_share.png"
plot_brand_market_share_comparison(brand_dict, fig_brand_path)
print(f"Đã xuất biểu đồ: {fig_brand_path}")

## 4. Phân phối theo Nhà bán lẻ (seller_raw)

In [ ]:
seller_df = get_seller_distribution(df)
display(seller_df)

fig_seller_path = "src/eda/outputs/figures/fig_1_1_seller_market_share.png"
plot_seller_share(seller_df, fig_seller_path)
print(f"Đã xuất biểu đồ: {fig_seller_path}")

## 5. Kết luận sơ bộ Nhiệm vụ 1.1

1. **Mất cân bằng nguồn cào (Source Asymmetry)**:
   * `cellphones` chiếm áp đảo với 76,07% tổng số listing (7.847 sản phẩm), tiếp theo là `tgdd` (11,60%) và `fptshop` (7,95%).
   * Khi phân tích các đặc trưng chi tiết (RAM, CPU, Pin) cần lưu ý xu hướng của CellphoneS có thể đại diện cho phần lớn dữ liệu.

2. **Cân bằng ngành hàng (Category Balance)**:
   * Tỷ lệ phân bố giữa hai danh mục khá đồng đều: **Smartphone** (5.442 mẫu - 52,75%) và **Laptop** (4.874 mẫu - 47,25%).

3. **Đặc trưng thị phần thương hiệu (Brand Concentration)**:
   * *Laptop*: Mức độ tập trung rất cao ở Top 5 thương hiệu: **ASUS (23,88%)**, **Apple (19,72%)**, **Lenovo (15,18%)**, **HP (10,73%)**, và **Dell (10,16%)** — chiếm tới gần 80% thị phần.
   * *Smartphone*: Cạnh tranh đa cực với 2 ông lớn dẫn đầu là **Samsung (17,92%)** và **Xiaomi (17,42%)**, kế tiếp là **Apple (13,27%)**, **OPPO (9,06%)**, **Vivo (6,03%)** và **Realme (5,26%)**.